# Lesson 30: Your data science roadmap · Practice

**Open Fraud Labs Academy**

1. Copy your **practice key** from the Practice tab of this lesson on the Academy and paste it into the first code cell.
2. Run the first cell, then complete each exercise and run its check.
3. Every check that passes is saved to your lesson page automatically. Solve all three to unlock the quiz.

Tip: choose **File > Save a copy in Drive** to keep your work.

In [ ]:
# Step 1: paste your practice key between the quotes, then run this cell (Shift + Enter).
# Find your key on the lesson page, in the Practice tab.
OFL_KEY = ""

import pandas as pd, requests
DATA_URL = "https://raw.githubusercontent.com/Odugbile1993/openfraudlab-tiktok/main/data/loans.csv"
COURSE, LESSON = "data-science", 30

def ofl_check(exercise, check_code):
    """Runs the check for one exercise and, if it passes, records it on your Academy lesson page."""
    try:
        exec(check_code, globals())
    except AssertionError as e:
        print("Not quite yet:", e or "compare your result with the task and try again.")
        return
    except NameError as e:
        print("Something the task asks for is missing. Run your code cell first.", e)
        return
    print("Correct! Well done.")
    if not OFL_KEY.strip():
        print("Add your practice key in the first cell and run it, then run this check again to record your progress.")
        return
    try:
        r = requests.post("https://virxrqwxvgsbcnmhdwlp.supabase.co/rest/v1/rpc/practice_report",
                          headers={"apikey": "sb_publishable_8IenXF_4F5ew8hLai8RfVA_dNHq4I72", "Content-Type": "application/json"},
                          json={"p_key": OFL_KEY.strip(), "p_course": COURSE, "p_lesson": LESSON, "p_exercise": exercise}, timeout=20)
        d = r.json()
        if r.ok:
            print(f"Saved to your lesson page: {d['solved']} of {d['total']} exercises solved.")
        else:
            print("Could not save your progress:", d.get("message", r.text))
    except Exception as e:
        print("Could not reach the Academy to save your progress:", e)

print("Ready. Work through the exercises below." if OFL_KEY.strip() else "Paste your practice key above so your progress is saved.")

## Exercise 1: Churn by contract

Store the churn rate for each contract type in rates (a Series indexed by contract), and how many times higher the Monthly rate is than the Annual rate in ratio.

<details><summary>Hint</summary>

subs.groupby("contract")["churned"].mean() gives one rate per contract. Divide rates["Monthly"] by rates["Annual"].

</details>

In [ ]:
import pandas as pd

subs = pd.read_csv(DATA_URL.replace("loans.csv", "global/subscribers.csv"))

rates = ...
ratio = ...
print(rates, ratio)

In [ ]:
# Check exercise 1 (run after your code above)
ofl_check(1, "_s = subs.copy() if \"subs\" in globals() and hasattr(subs, \"copy\") else None\nimport pandas as _pd\n_s = _pd.read_csv(DATA_URL.replace(\"loans.csv\", \"global/subscribers.csv\"))\n_s[\"avg_weekly_hours\"] = _s[\"avg_weekly_hours\"].fillna(_s[\"avg_weekly_hours\"].median())\n_s[\"monthly_contract\"] = _s[\"contract\"].map({\"Annual\": 0, \"Monthly\": 1})\n_base = [\"age\", \"monthly_fee_usd\", \"tenure_months\", \"avg_weekly_hours\", \"devices\", \"support_tickets_90d\", \"auto_renew\", \"monthly_contract\"]\n_r = _s.groupby(\"contract\")[\"churned\"].mean()\nassert isinstance(rates, _pd.Series) and sorted(rates.index) == sorted(_r.index), \"rates should be subs.groupby(\\\"contract\\\")[\\\"churned\\\"].mean()\"\nassert all(abs(float(rates[k]) - _r[k]) < 1e-9 for k in _r.index), \"rates should hold the churn rate of each contract type\"\nassert ratio is not Ellipsis and abs(float(ratio) - _r[\"Monthly\"] / _r[\"Annual\"]) < 1e-9, \"ratio should be rates[\\\"Monthly\\\"] / rates[\\\"Annual\\\"]\"\n")

## Exercise 2: End to end

Split X and y (test_size=0.2, random_state=42, stratify=y). Build a pipeline of SimpleImputer(strategy="median"), StandardScaler() and LogisticRegression(max_iter=1000, class_weight="balanced"), fit it on the training data, predict the test set, and store the recall in rec and the precision in prec.

<details><summary>Hint</summary>

make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")). Then fit, predict and score.

</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score

subs = pd.read_csv(DATA_URL.replace("loans.csv", "global/subscribers.csv"))
subs["monthly_contract"] = subs["contract"].map({"Annual": 0, "Monthly": 1})
features = ["age", "monthly_fee_usd", "tenure_months", "avg_weekly_hours",
            "devices", "support_tickets_90d", "auto_renew", "monthly_contract"]
X, y = subs[features], subs["churned"]

X_train, X_test, y_train, y_test = ...
model = ...
# fit, predict, score
rec = ...
prec = ...
print(rec, prec)

In [ ]:
# Check exercise 2 (run after your code above)
ofl_check(2, "import pandas as _pd\nfrom sklearn.pipeline import make_pipeline as _mp\nfrom sklearn.impute import SimpleImputer as _SI\nfrom sklearn.preprocessing import StandardScaler as _SS\nfrom sklearn.linear_model import LogisticRegression as _LR\n_s = _pd.read_csv(DATA_URL.replace(\"loans.csv\", \"global/subscribers.csv\"))\n_s[\"monthly_contract\"] = _s[\"contract\"].map({\"Annual\": 0, \"Monthly\": 1})\n_base = [\"age\", \"monthly_fee_usd\", \"tenure_months\", \"avg_weekly_hours\", \"devices\", \"support_tickets_90d\", \"auto_renew\", \"monthly_contract\"]\n_mk = lambda: _mp(_SI(strategy=\"median\"), _SS(), _LR(max_iter=1000, class_weight=\"balanced\"))\nfrom sklearn.model_selection import train_test_split as _tts\nfrom sklearn.metrics import recall_score as _rs, precision_score as _ps\n_Xtr, _Xte, _ytr, _yte = _tts(_s[_base], _s[\"churned\"], test_size=0.2, random_state=42, stratify=_s[\"churned\"])\nassert list(X_test.index) == list(_Xte.index), \"Split with train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)\"\n_p = _mk().fit(_Xtr, _ytr).predict(_Xte)\nassert rec is not Ellipsis and abs(float(rec) - _rs(_yte, _p)) < 1e-9, \"rec should be recall_score(y_test, preds) after fitting the three-step pipeline\"\nassert prec is not Ellipsis and abs(float(prec) - _ps(_yte, _p)) < 1e-9, \"prec should be precision_score(y_test, preds)\"\n")

## Exercise 3: Does the feature earn its place?

Using the model and folds provided, cross-validate recall on all subscribers twice: with the base features, and with base plus fee_per_hour. Store the two mean recalls in without_fph and with_fph.

<details><summary>Hint</summary>

cross_val_score(model, subs[base], y, cv=cv, scoring="recall").mean(); then the same with subs[base + ["fee_per_hour"]].

</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

subs = pd.read_csv(DATA_URL.replace("loans.csv", "global/subscribers.csv"))
subs["monthly_contract"] = subs["contract"].map({"Annual": 0, "Monthly": 1})
subs["fee_per_hour"] = subs["monthly_fee_usd"] / subs["avg_weekly_hours"]
base = ["age", "monthly_fee_usd", "tenure_months", "avg_weekly_hours",
        "devices", "support_tickets_90d", "auto_renew", "monthly_contract"]
y = subs["churned"]
model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                      LogisticRegression(max_iter=1000, class_weight="balanced"))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

without_fph = ...
with_fph = ...
print(without_fph, with_fph)

In [ ]:
# Check exercise 3 (run after your code above)
ofl_check(3, "import pandas as _pd\nfrom sklearn.pipeline import make_pipeline as _mp\nfrom sklearn.impute import SimpleImputer as _SI\nfrom sklearn.preprocessing import StandardScaler as _SS\nfrom sklearn.linear_model import LogisticRegression as _LR\n_s = _pd.read_csv(DATA_URL.replace(\"loans.csv\", \"global/subscribers.csv\"))\n_s[\"monthly_contract\"] = _s[\"contract\"].map({\"Annual\": 0, \"Monthly\": 1})\n_base = [\"age\", \"monthly_fee_usd\", \"tenure_months\", \"avg_weekly_hours\", \"devices\", \"support_tickets_90d\", \"auto_renew\", \"monthly_contract\"]\n_mk = lambda: _mp(_SI(strategy=\"median\"), _SS(), _LR(max_iter=1000, class_weight=\"balanced\"))\nfrom sklearn.model_selection import StratifiedKFold as _SKF, cross_val_score as _cvs\n_s[\"fee_per_hour\"] = _s[\"monthly_fee_usd\"] / _s[\"avg_weekly_hours\"]\n_cv = _SKF(n_splits=5, shuffle=True, random_state=42)\n_a = _cvs(_mk(), _s[_base], _s[\"churned\"], cv=_cv, scoring=\"recall\").mean()\n_c = _cvs(_mk(), _s[_base + [\"fee_per_hour\"]], _s[\"churned\"], cv=_cv, scoring=\"recall\").mean()\nassert isinstance(without_fph, float) and abs(without_fph - _a) < 1e-9, \"without_fph should be cross_val_score(model, subs[base], y, cv=cv, scoring=\\\"recall\\\").mean()\"\nassert isinstance(with_fph, float) and abs(with_fph - _c) < 1e-9, \"with_fph should use subs[base + [\\\"fee_per_hour\\\"]]\"\n")

---
Done? Go back to the lesson on [openfraudlabs.com/academy](https://openfraudlabs.com/academy/lesson/?course=data-science&n=30) and take the quiz.